# Customer Segmentation & Behavioral Analysis

**Purpose**: Segment customers based on transaction behavior and value

**Business Questions**:
- Who are our most valuable customers?
- Can we identify customer segments for targeted marketing?
- Which customers are at risk of churn?
- What is customer lifetime value (CLV) distribution?
- How can we use RFM analysis for segmentation?

**Target Audience**: CMO, Product Management, Customer Success

In [ ]:
# Import libraries
import sys
sys.path.append('/home/jovyan/work')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from middleware import get_db, query, TransactionAnalytics, DataTransformers

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
%matplotlib inline

print("✓ Libraries imported successfully")

## 1. Load Customer and Transaction Data

In [ ]:
# Load customer data
customers = query("""
    SELECT customer_id, first_name, last_name, email, is_vip, vip_discount, 
           risk_level, created_date
    FROM customers
""")

# Load approved transactions only
transactions = query("""
    SELECT customer_id, amount, created_date, merchant_id
    FROM transactions
    WHERE status = 1
""")

transactions['created_date'] = pd.to_datetime(transactions['created_date'])

print(f"Loaded {len(customers):,} customers")
print(f"Loaded {len(transactions):,} approved transactions")

## 2. Calculate Customer Metrics

In [ ]:
# Calculate customer-level metrics using our utility
customer_metrics = TransactionAnalytics.calculate_customer_metrics(transactions)

# Merge with customer data
customer_analysis = customers.merge(customer_metrics, on='customer_id', how='left')

# Fill NaN for customers with no transactions
customer_analysis.fillna({
    'total_transactions': 0,
    'total_spent': 0,
    'avg_transaction': 0,
    'approval_rate': 0
}, inplace=True)

# Filter to customers with at least one transaction
active_customers = customer_analysis[customer_analysis['total_transactions'] > 0].copy()

print(f"Active customers (with transactions): {len(active_customers):,}")
print(f"\nSample metrics:")
active_customers.head()

## 3. RFM Analysis (Recency, Frequency, Monetary)

In [ ]:
# Apply RFM segmentation
rfm_customers = DataTransformers.create_customer_segments(active_customers)

# Segment distribution
segment_counts = rfm_customers['segment'].value_counts()

print("Customer Segments:")
for segment, count in segment_counts.items():
    pct = count / len(rfm_customers) * 100
    print(f"  {segment:25s}: {count:>6,} ({pct:>5.1f}%)")

# Plot segment distribution
plt.figure(figsize=(12, 6))
colors = sns.color_palette("Set2", len(segment_counts))
plt.bar(range(len(segment_counts)), segment_counts.values, color=colors, alpha=0.8)
plt.xticks(range(len(segment_counts)), segment_counts.index, rotation=45, ha='right')
plt.ylabel('Number of Customers')
plt.title('Customer Segmentation Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Segment characteristics
segment_summary = rfm_customers.groupby('segment').agg({
    'customer_id': 'count',
    'total_spent': 'mean',
    'total_transactions': 'mean',
    'days_since_last_transaction': 'mean',
    'avg_transaction': 'mean'
}).round(2)

segment_summary.columns = ['Count', 'Avg Total Spent', 'Avg Transactions', 'Avg Days Since Last', 'Avg Transaction']
segment_summary = segment_summary.sort_values('Avg Total Spent', ascending=False)

print("\nSegment Characteristics:")
print(segment_summary)

## 4. Customer Lifetime Value (CLV)

In [ ]:
# Simple CLV calculation
# CLV = (Avg Transaction Value) × (Transactions per Year) × (Customer Lifetime Years)

# Estimate transaction frequency (transactions per day * 365)
rfm_customers['txn_frequency'] = rfm_customers['total_transactions'] / (rfm_customers['customer_lifetime_days'] + 1)
rfm_customers['annual_transactions'] = rfm_customers['txn_frequency'] * 365

# Assume average customer lifetime of 3 years
avg_customer_lifetime_years = 3

rfm_customers['estimated_clv'] = (
    rfm_customers['avg_transaction'] * 
    rfm_customers['annual_transactions'] * 
    avg_customer_lifetime_years
)

# CLV distribution
plt.figure(figsize=(12, 6))
plt.hist(rfm_customers['estimated_clv'], bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
plt.xlabel('Estimated Customer Lifetime Value ($)')
plt.ylabel('Number of Customers')
plt.title('Customer Lifetime Value Distribution', fontsize=14, fontweight='bold')
plt.axvline(rfm_customers['estimated_clv'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: ${rfm_customers["estimated_clv"].median():,.0f}')
plt.axvline(rfm_customers['estimated_clv'].mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean: ${rfm_customers["estimated_clv"].mean():,.0f}')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average CLV: ${rfm_customers['estimated_clv'].mean():,.2f}")
print(f"Median CLV: ${rfm_customers['estimated_clv'].median():,.2f}")
print(f"Total customer base value: ${rfm_customers['estimated_clv'].sum():,.2f}")

## 5. High-Value Customers (VIP Analysis)

In [ ]:
# Identify top 10% by total spent
top_10_pct_threshold = rfm_customers['total_spent'].quantile(0.90)
high_value_customers = rfm_customers[rfm_customers['total_spent'] >= top_10_pct_threshold]

print(f"High-value customer threshold: ${top_10_pct_threshold:,.2f}")
print(f"High-value customers: {len(high_value_customers):,} ({len(high_value_customers)/len(rfm_customers)*100:.1f}%)")
print(f"Revenue from high-value customers: ${high_value_customers['total_spent'].sum():,.2f}")
print(f"Percentage of total revenue: {high_value_customers['total_spent'].sum()/rfm_customers['total_spent'].sum()*100:.1f}%")

# Compare VIP status vs actual value
vip_analysis = rfm_customers.groupby('is_vip').agg({
    'customer_id': 'count',
    'total_spent': 'mean',
    'total_transactions': 'mean',
    'estimated_clv': 'mean'
}).round(2)

vip_analysis.columns = ['Count', 'Avg Spent', 'Avg Transactions', 'Avg CLV']
print("\nVIP vs Non-VIP Comparison:")
print(vip_analysis)

## 6. Churn Risk Analysis

In [ ]:
# Customers who haven't transacted in 90+ days are at risk
churn_risk_days = 90

at_risk_customers = rfm_customers[rfm_customers['days_since_last_transaction'] >= churn_risk_days]
high_value_at_risk = at_risk_customers[at_risk_customers['total_spent'] >= top_10_pct_threshold]

print(f"Customers at churn risk (no activity in {churn_risk_days}+ days): {len(at_risk_customers):,}")
print(f"High-value customers at risk: {len(high_value_at_risk):,}")
print(f"Potential revenue loss: ${high_value_at_risk['total_spent'].sum():,.2f}")

# At-risk customers by segment
at_risk_segments = at_risk_customers['segment'].value_counts()

plt.figure(figsize=(10, 6))
plt.bar(range(len(at_risk_segments)), at_risk_segments.values, color='#D72638', alpha=0.7)
plt.xticks(range(len(at_risk_segments)), at_risk_segments.index, rotation=45, ha='right')
plt.ylabel('Number of At-Risk Customers')
plt.title(f'Churn Risk by Segment ({churn_risk_days}+ Days Inactive)', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. K-Means Clustering

In [ ]:
# Prepare features for clustering
clustering_features = rfm_customers[[
    'total_spent',
    'total_transactions',
    'days_since_last_transaction'
]].copy()

# Scale features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(clustering_features)

# Determine optimal number of clusters (Elbow method)
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, marker='o', linewidth=2)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Use k=4 clusters
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm_customers['cluster'] = kmeans.fit_predict(features_scaled)

print(f"\nUsing {optimal_k} clusters")
print("Cluster distribution:")
print(rfm_customers['cluster'].value_counts().sort_index())

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Total Spent vs Transactions
scatter1 = axes[0].scatter(
    rfm_customers['total_transactions'],
    rfm_customers['total_spent'],
    c=rfm_customers['cluster'],
    cmap='viridis',
    alpha=0.6,
    s=50
)
axes[0].set_xlabel('Total Transactions')
axes[0].set_ylabel('Total Spent ($)')
axes[0].set_title('Clusters: Spending vs Transaction Count')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Total Spent vs Recency
scatter2 = axes[1].scatter(
    rfm_customers['days_since_last_transaction'],
    rfm_customers['total_spent'],
    c=rfm_customers['cluster'],
    cmap='viridis',
    alpha=0.6,
    s=50
)
axes[1].set_xlabel('Days Since Last Transaction')
axes[1].set_ylabel('Total Spent ($)')
axes[1].set_title('Clusters: Spending vs Recency')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

# Cluster profiles
cluster_profiles = rfm_customers.groupby('cluster').agg({
    'customer_id': 'count',
    'total_spent': 'mean',
    'total_transactions': 'mean',
    'days_since_last_transaction': 'mean',
    'estimated_clv': 'mean'
}).round(2)

cluster_profiles.columns = ['Count', 'Avg Spent', 'Avg Transactions', 'Avg Recency', 'Avg CLV']
print("\nCluster Profiles:")
print(cluster_profiles)

## 8. Actionable Customer Lists

In [ ]:
# Export customer lists for marketing campaigns

# 1. Champions for upsell
champions = rfm_customers[rfm_customers['segment'] == 'Champions'][[
    'customer_id', 'email', 'total_spent', 'total_transactions', 'estimated_clv'
]].head(100)

# 2. At-risk high-value for retention
retention_targets = high_value_at_risk[[
    'customer_id', 'email', 'total_spent', 'days_since_last_transaction'
]].head(100)

# 3. New customers for onboarding
new_customers = rfm_customers[rfm_customers['customer_lifetime_days'] <= 30][[
    'customer_id', 'email', 'total_spent', 'total_transactions'
]].head(100)

print("Customer Lists Generated:")
print(f"  Champions (for upsell): {len(champions)}")
print(f"  High-value at risk (for retention): {len(retention_targets)}")
print(f"  New customers (for onboarding): {len(new_customers)}")

# Could export to CSV here:
# champions.to_csv('/home/jovyan/work/data/champions.csv', index=False)
# retention_targets.to_csv('/home/jovyan/work/data/retention_targets.csv', index=False)
# new_customers.to_csv('/home/jovyan/work/data/new_customers.csv', index=False)

## 9. Summary & Recommendations

### Key Findings:

**Customer Value:**
- Top 10% of customers drive significant revenue share
- Average CLV provides baseline for acquisition cost decisions
- VIP status correlates with higher spending

**Segmentation:**
- Clear segments identified: Champions, Loyal, At-Risk, Lost
- K-means clustering reveals natural customer groupings
- Different segments require different strategies

**Churn Risk:**
- Significant number of high-value customers at risk
- 90+ days inactivity is strong churn indicator
- Lost customers segment needs win-back campaigns

### Recommendations:

1. **Retention Programs:**
   - Launch re-engagement campaign for at-risk high-value customers
   - Implement automated alerts for VIP inactivity
   - Create loyalty rewards for Champions

2. **Marketing Strategy:**
   - Segment-specific messaging and offers
   - Focus acquisition on high-CLV profiles
   - Increase touchpoints with Potential Loyalists

3. **Product Development:**
   - Design features to increase transaction frequency
   - Create VIP-exclusive benefits
   - Improve onboarding for new customers

4. **Analytics:**
   - Regular CLV recalculation (monthly)
   - Churn prediction model development
   - Cohort analysis for retention tracking